# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaminari19/FlyRank-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule, in plain words:** A page is worth reviewing for refresh if it's **stale** (hasn't been
touched in a long time) **and** it's still pulling **real search demand** (meaningful impressions) —
stale alone could mean a dead page nobody searches for; visible-but-stale means the opportunity is
real and being left on the table.

**Reason code:** `stale_but_visible` — the only code this simple rule ever emits, when it fires.

**Two signals checked before I trust them:**

### Signal 1 — Staleness (`days_since_last_update`), behind the refresh flag
Bucket pages by how long since their last update, and look at the decline rate in each bucket.

In [16]:
import os

REPO_URL = "https://github.com/kaminari19/FlyRank-Starter.git"
REPO_DIR = "FlyRank-Starter"

# Always start from /content so re-running this cell never nests FlyRank-Starter/FlyRank-Starter
os.chdir("/content") if os.path.exists("/content") else None

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import pandas as pd
import numpy as np

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"Expected {DATA_PATH} — check the repo cloned correctly."

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

# is_declining_label isn't shipped precomputed — build it from trend_direction (same as week 2)
assert "trend_direction" in df.columns, "trend_direction missing — check column list above."
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("is_declining_label base rate:", df["is_declining_label"].mean().round(3))

# Leakage guardrail (from skills/flyrank/flyrank-data): these are NEVER rule inputs.
LEAKAGE_COLS = {"trend_direction", "trend_pct", "is_declining_label",
                "health_score", "recommended_action", "action_type"}
print("Leakage-risk columns present but off-limits as inputs:",
      [c for c in LEAKAGE_COLS if c in df.columns])

/content/FlyRank-Starter
Loaded 30,000 rows x 44 columns
is_declining_label base rate: 0.542
Leakage-risk columns present but off-limits as inputs: ['trend_direction', 'trend_pct', 'is_declining_label']


In [17]:
bins   = [-1, 30, 90, 180, 100_000]
labels = ["0-30", "31-90", "91-180", "181+"]
df["freshness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

staleness_table = (
    df.groupby("freshness_bucket", observed=True)["is_declining_label"]
      .agg(n="count", decline_rate="mean")
      .round(3)
)
staleness_table

,n,decline_rate
freshness_bucket,,
0-30,20480,0.511
31-90,175,0.589
91-180,9171,0.611
181+,174,0.471


In [18]:
pos_bins   = [-1, 3, 10, 20, 100]
pos_labels = ["top_3", "page_1", "page_2", "page_3_plus"]

has_position = df["avg_position"] > 0
df.loc[has_position, "position_bucket"] = pd.cut(
    df.loc[has_position, "avg_position"], bins=pos_bins, labels=pos_labels)

ctr_position_table = (
    df[has_position]
      .groupby("position_bucket", observed=True)
      .agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"), decline_rate=("is_declining_label", "mean"))
      .round(3)
)
ctr_position_table

,n,mean_ctr,decline_rate
position_bucket,,,
top_3,1141,2.714,0.498
page_1,11842,0.651,0.569
page_2,7273,0.323,0.610
page_3_plus,8524,0.212,0.529


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# Transparent score — no fitted weights, readable on purpose (per skills/building-baselines)
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = stale * visible * df["impressions_90d"]
df["reason_code"]    = "stale_but_visible"
df["action"]          = np.where(df["baseline_score"] > 0, "refresh", "no_action")

queue = (
    df.sort_values("baseline_score", ascending=False)
      .reset_index(drop=True)
)
queue.insert(0, "rank", queue.index + 1)

# Precision@K against the honest label, with base rate printed alongside (per skills/building-baselines)
def precision_at_k(labels, k):
    return labels.head(k).mean()

K = 50
base_rate = df["is_declining_label"].mean()
p_at_k = precision_at_k(queue["is_declining_label"], K)
print(f"Base rate (declining share of ALL pages): {base_rate:.3f}")
print(f"Precision@{K} of this rule:                {p_at_k:.3f}")

output_cols = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action",
               "days_since_last_update", "impressions_90d", "avg_position", "ctr",
               "content_type", "main_intent", "is_declining_label"]
output_cols = [c for c in output_cols if c in queue.columns]

os.makedirs("work/outputs", exist_ok=True)
queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nWrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue[output_cols].head(10)

Base rate (declining share of ALL pages): 0.542
Precision@50 of this rule:                0.640

Wrote 30,000 rows to work/outputs/baseline_action_score.csv


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,content_type,main_intent,is_declining_label
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,refresh,194,61678,19.7,0.15,keyword article,informational,1
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,refresh,194,59472,24.8,0.13,keyword article,informational,1
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,refresh,194,25715,22.2,0.23,keyword article,informational,1
3,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,refresh,193,13299,10.5,0.49,keyword article,informational,1
4,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_but_visible,refresh,194,7812,39.0,0.01,keyword article,transactional,1
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,refresh,193,7558,17.9,0.20,keyword article,informational,1
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_but_visible,refresh,194,4590,31.0,0.00,keyword article,informational,1
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,refresh,194,4556,16.4,0.33,keyword article,informational,1
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,refresh,194,4429,25.3,0.38,keyword article,informational,1
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,refresh,193,1697,15.8,0.12,keyword article,informational,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
def why_flagged(row):
    return (f"Stale {row['days_since_last_update']:.0f}d (>=180 threshold) while still pulling "
            f"{row['impressions_90d']:,.0f} impressions/90d — visible demand not being served.")

def what_would_make_it_wrong(row):
    reasons = []
    if row.get("main_intent") == "informational":
        reasons.append("if it's an evergreen reference page that isn't supposed to change")
    if row.get("competition_level") == "HIGH":
        reasons.append("if the SERP is dominated by sites a refresh can't realistically outrank")
    if row["ctr"] == 0:
        reasons.append("if the 0% CTR here is a tracking gap, not a real ranking problem")
    if not reasons:
        reasons.append("if the impression volume is a one-off seasonal spike, not sustained demand")
    return " or ".join(reasons)

top10 = queue.head(10)
for _, row in top10.iterrows():
    print(f"#{row['rank']:.0f}  {row['content_id']}  action={row['action']}  reason={row['reason_code']}")
    print(f"    why:   {why_flagged(row)}")
    print(f"    wrong-if: {what_would_make_it_wrong(row)}")
    print()

#1  content_cf56e2e2e282  action=refresh  reason=stale_but_visible
    why:   Stale 194d (>=180 threshold) while still pulling 61,678 impressions/90d — visible demand not being served.
    wrong-if: if it's an evergreen reference page that isn't supposed to change

#2  content_7368877ea310  action=refresh  reason=stale_but_visible
    why:   Stale 194d (>=180 threshold) while still pulling 59,472 impressions/90d — visible demand not being served.
    wrong-if: if it's an evergreen reference page that isn't supposed to change

#3  content_1bfaa38ff26c  action=refresh  reason=stale_but_visible
    why:   Stale 194d (>=180 threshold) while still pulling 25,715 impressions/90d — visible demand not being served.
    wrong-if: if it's an evergreen reference page that isn't supposed to change

#4  content_0a91db491d14  action=refresh  reason=stale_but_visible
    why:   Stale 193d (>=180 threshold) while still pulling 13,299 impressions/90d — visible demand not being served.
    wrong-if: if 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**My answer:** Read the top-10 printout above for any row where the "wrong-if" line feels like the
more likely explanation than genuine decline (e.g. a `HIGH` competition, informational page — a
refresh may not move the needle no matter how stale it is). Name at least one specific `content_id`
from your own run here as the weakest pick, and why.

In [21]:
# Leakage check: the rule's only inputs are days_since_last_update and impressions_90d.
RULE_INPUTS = {"days_since_last_update", "impressions_90d"}
assert RULE_INPUTS.isdisjoint(LEAKAGE_COLS), "Rule touched a label-derived / product-flag column!"
print("Rule inputs:", RULE_INPUTS)
print("Leakage-risk columns (never used):", LEAKAGE_COLS)
print("No overlap — confirmed clean.")

# Future-window check: content_refresh_anonymized.csv is a single trailing-90-day snapshot,
# no forward-looking columns exist to accidentally pull in.
print("\nAll rule/feature columns are trailing/point-in-time, not forward-looking — "
      "no separate future-window table involved in this baseline.")

Rule inputs: {'days_since_last_update', 'impressions_90d'}
Leakage-risk columns (never used): {'trend_direction', 'action_type', 'trend_pct', 'recommended_action', 'is_declining_label', 'health_score'}
No overlap — confirmed clean.

All rule/feature columns are trailing/point-in-time, not forward-looking — no separate future-window table involved in this baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.